# 10 · Query: top-k results and scores
Level L2 · Part 2 · builds on lesson 09 (and the scores of lessons 04 and 08)

## Goal
By the end of this lesson you can run a similarity query with `query_points`, choose how many
results come back (`limit`), read each result's score, and cut weak results off with a
`score_threshold` chosen from data, in the right direction for the collection's metric.

## The idea
A **query** hands the database one vector and asks: which stored points are closest? The answer
is a ranked list of **scored points**: id, payload and a **score** (for Cosine, the cosine
similarity). `limit` is the k of top-k: the most results you want. But top-k always fills up:
ask for 10 and you get 10, even when nothing fits. A **score threshold** adds a second rule:
"and only if the score is at least this good". Think of a dating app: `limit` is "show me 5
people", the threshold is "only people who match at least 60%".

## Picture

```mermaid
flowchart LR
    Q["query vector<br/>'comfort food for<br/>a rainy evening'"] --> S["score all 50 points<br/>(Cosine)"]
    S --> R["rank by score<br/>0.537, 0.435, 0.422, ..."]
    R --> T{"score_threshold?<br/>drop weak scores"}
    T --> L["limit = k<br/>keep the first k"]
    L --> A["QueryResponse.points<br/>[ScoredPoint(id, score, payload), ...]"]
```

Both rules cut the same ranked list: the threshold by quality, the limit by count. Whichever
is stricter decides how many results you get.

## Step by step
### Step 1 · The `recipes` collection again
Each lesson has its own local folder, so we rebuild lesson 09's collection: 50 points, MiniLM
vectors (384, Cosine) and the same payload. We also load the cached query vectors from lesson 06
(15 queries embedded with the same model), so no model download is needed.

In [1]:
import shutil, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_50.csv")
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
cache = np.load("../data/query_embeddings_minilm.npz")
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"]))
print("model:", MODEL, "| recipes:", doc_emb.shape, "| cached queries:", len(query_cache))

model: sentence-transformers/all-MiniLM-L6-v2 | recipes: (50, 384) | cached queries: 15


Now the collection and the points, written exactly as in lesson 09 (`to_payload` turns a CSV row
into plain JSON types).

In [2]:
DB_PATH = "../data/cache/qdrant_10"
shutil.rmtree(DB_PATH, ignore_errors=True)           # start from an empty folder every run
client = QdrantClient(path=DB_PATH)
client.create_collection("recipes", vectors_config=models.VectorParams(
    size=doc_emb.shape[1], distance=models.Distance.COSINE))

def to_payload(row):
    return {"title": str(row.title), "cuisine": str(row.cuisine), "vegetarian": bool(row.vegetarian),
            "minutes": int(row.minutes), "embed_model": MODEL}

points = [models.PointStruct(id=int(row.id), vector=doc_emb[i].tolist(), payload=to_payload(row))
          for i, row in enumerate(recipes.itertuples())]
client.upsert(collection_name="recipes", points=points)
print("points in collection:", client.count("recipes").count)

points in collection: 50


### Step 2 · The smallest query
`query_points` (you met it briefly in lesson 08) takes the collection name, the query vector and `limit`. Ask for the 3 recipes
closest to "comfort food for a rainy evening":

In [3]:
q = query_cache["comfort food for a rainy evening"]
response = client.query_points(collection_name="recipes", query=q.tolist(), limit=3)
print(type(response).__name__, "with", len(response.points), "points")
for hit in response.points:
    print(f"  id {hit.id:>2}  score {hit.score:.3f}  {hit.payload['title']}")

QueryResponse with 3 points
  id  7  score 0.537  Chicken noodle soup
  id 50  score 0.435  Gazpacho
  id 49  score 0.422  Ramen


The call returns a `QueryResponse`; the ranked hits are in its `.points` list, best first.
Chicken noodle soup wins with 0.537. Each hit is a `ScoredPoint`. Look at one whole:

In [4]:
top = response.points[0]
print(top)
print("fields:", list(top.model_dump().keys()))

id=7 version=0 score=0.5366013885410725 payload={'title': 'Chicken noodle soup', 'cuisine': 'American', 'vegetarian': False, 'minutes': 40, 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2'} vector=None shard_key=None order_value=None
fields: ['id', 'version', 'score', 'payload', 'vector', 'shard_key', 'order_value']


A `ScoredPoint` is a lesson 09 `Record` plus a `score`. Like `retrieve`, the payload comes back
and the vector does not (`vector=None`). `version` is a write counter (always 0 in local mode);
`shard_key` and `order_value` stay `None` until later lessons. Leave out `limit` and you get the
default of 10; trim the payload with `with_payload`, exactly as in lesson 09:

In [5]:
default = client.query_points("recipes", query=q.tolist(), with_payload=["title"])
print("no limit given ->", len(default.points), "results")
print(default.points[3])

no limit given -> 10 results
id=47 version=0 score=0.41140328078118976 payload={'title': 'Hummus'} vector=None shard_key=None order_value=None


### Step 3 · What the score is
On a Cosine collection the score is the cosine similarity between the query and the stored
vector. MiniLM vectors have length 1, so that is just the dot product (lesson 03). Check the top
3 against NumPy:

In [6]:
row_of = {int(i): n for n, i in enumerate(recipes["id"])}       # point id -> row in doc_emb
for hit in response.points:
    by_hand = float(doc_emb[row_of[hit.id]] @ q)
    print(f"{hit.payload['title']:<20} Qdrant {hit.score:.4f}   NumPy dot {by_hand:.4f}")

Chicken noodle soup  Qdrant 0.5366   NumPy dot 0.5366
Gazpacho             Qdrant 0.4348   NumPy dot 0.4348
Ramen                Qdrant 0.4222   NumPy dot 0.4222


Identical to 4 decimals. A score is a number about **two vectors**, not a percentage of
"relevance". Every stored point gets one, however unrelated. Ask for all 50:

In [7]:
everything = client.query_points("recipes", query=q.tolist(), limit=50).points
scores = np.array([h.score for h in everything])
for rank in [1, 2, 25, 49, 50]:
    h = everything[rank - 1]
    print(f"rank {rank:>2}: {h.score:.3f}  {h.payload['title']}")
print("asked for 100, got:", len(client.query_points("recipes", query=q.tolist(), limit=100).points))

rank  1: 0.537  Chicken noodle soup
rank  2: 0.435  Gazpacho
rank 25: 0.360  Pesto pasta
rank 49: 0.191  Apple pie
rank 50: 0.100  Strawberry cheesecake
asked for 100, got: 50


Even the last recipe has a score (0.100). Search never says "no match" on its own: it ranks.
And `limit` larger than the collection simply returns everything (50).

### Step 4 · Choosing k
`limit` is the k of top-k. The search here is exact (local mode, lesson 07), so a smaller k is
always the start of a larger k's list:

In [8]:
for k in [1, 3, 5]:
    hits = client.query_points("recipes", query=q.tolist(), limit=k).points
    print(f"k={k}:", [h.payload["title"] for h in hits])

k=1: ['Chicken noodle soup']
k=3: ['Chicken noodle soup', 'Gazpacho', 'Ramen']
k=5: ['Chicken noodle soup', 'Gazpacho', 'Ramen', 'Hummus', 'French onion soup']


Choose k from what the result is *for*: 1 for "the answer", 3 to 10 for a list a person scans,
20 to 100 when another step re-sorts them later (reranking, lesson 21; RAG, lesson 25). Bigger k
costs more data sent back, and in Part 4's approximate indexes, more search work too.

### Step 5 · Query with a stored point
The query can also be the **id** of a point already in the collection. Qdrant uses that
point's vector as the query: "more like recipe 1 (Tomato soup)".

In [9]:
like_soup = client.query_points("recipes", query=1, limit=3).points
print([(h.id, h.payload["title"], round(h.score, 3)) for h in like_soup])
print("Tomato soup itself in the results?", any(h.id == 1 for h in like_soup))

[(9, 'French onion soup', 0.712), (6, 'Minestrone', 0.687), (50, 'Gazpacho', 0.625)]
Tomato soup itself in the results? False


Three other soups, and here (local mode) Tomato soup itself is left out; check this on your
server version. Handy for a "similar recipes" box; lesson
24 grows it into recommendation queries ("more like these, less like that").

### Step 6 · Are scores comparable between queries?
A threshold only works if "good" scores and "bad" scores fall in different ranges. Run all 15
cached queries and print the top hit's score:

In [10]:
rows = []
for text, vec in query_cache.items():
    best = client.query_points("recipes", query=vec.tolist(), limit=1).points[0]
    rows.append((text, best.payload["title"], round(best.score, 3)))
top1 = pd.DataFrame(rows, columns=["query", "top hit", "score"]).sort_values("score", ascending=False)
print(top1.to_string(index=False))

                           query                top hit  score
                   Italian pasta    Spaghetti carbonara  0.706
                 very spicy food    Spicy tofu stir-fry  0.634
                   healthy salad           Caesar salad  0.627
       a cold dessert for summer           Lemon sorbet  0.603
 something to eat in the morning Full English breakfast  0.593
                     a hot curry               Vindaloo  0.592
         quick vegetarian dinner              Beef stew  0.559
                meat-free dinner              Beef stew  0.547
      a fancy dessert for guests  Strawberry cheesecake  0.543
comfort food for a rainy evening    Chicken noodle soup  0.537
        something with chickpeas           Falafel wrap  0.514
                         seafood         Grilled salmon  0.502
                 a dip for chips Chocolate chip cookies  0.500
                 a chilled drink            Mango lassi  0.469
   something warm for a cold day    Chicken noodle soup

Read the table, not just the numbers. "quick vegetarian dinner" → Beef stew (0.559) is **wrong**
yet outscores "seafood" → Grilled salmon (0.502), which is right. "a dip for chips" → Chocolate
chip cookies (0.500) is wrong too: the word "chip" matched. Each query has its own score scale,
so no threshold can separate right from wrong answers among on-topic queries.

### Step 7 · Queries the recipes cannot answer
What a threshold *can* catch is a query that has nothing to do with cooking. We embed four
off-topic questions with the same model and save them (needs the model download; the saved file
is in `data/`, so the next cell works without it).

In [11]:
import os
os.environ.update(TQDM_DISABLE="1", HF_HUB_VERBOSITY="error")
from sentence_transformers import SentenceTransformer

off_topic = ["how to change a car tyre", "quarterly tax return deadline",
             "learn to play the guitar", "the history of the Roman empire"]
model = SentenceTransformer(MODEL)
np.savez("../data/query_embeddings_minilm_offtopic.npz", model=MODEL,
         queries=np.array(off_topic), query_emb=model.encode_query(off_topic))
print("saved", len(off_topic), "off-topic query vectors")

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Load the saved vectors (written by the cell above, or shipped with the course) and look at the
best score each one gets:

In [12]:
extra = np.load("../data/query_embeddings_minilm_offtopic.npz")
assert str(extra["model"]) == MODEL                   # same model as the recipe vectors
off_cache = dict(zip(extra["queries"].tolist(), extra["query_emb"]))
for text, vec in off_cache.items():
    best = client.query_points("recipes", query=vec.tolist(), limit=1).points[0]
    print(f"{text:<34} -> {best.payload['title']:<15} {best.score:.3f}")

how to change a car tyre           -> Overnight oats  0.111
quarterly tax return deadline      -> Falafel wrap    0.073
learn to play the guitar           -> Fish tacos      0.114
the history of the Roman empire    -> Caesar salad    0.237


Without a threshold, a guitar question returns Fish tacos. These top scores (0.07 to 0.24) are
well below every on-topic query in Step 6 (lowest 0.444). That gap is where a threshold belongs.

### Step 8 · Add `score_threshold`
`score_threshold` drops every hit whose score is worse than the value. Use 0.3, inside the gap,
with `limit=5`, on two on-topic and two off-topic queries:

In [13]:
def search(vec, k=5, min_score=None):
    hits = client.query_points("recipes", query=vec.tolist(), limit=k, score_threshold=min_score).points
    return [(h.payload["title"], round(h.score, 3)) for h in hits]

for text, vec in [("seafood", query_cache["seafood"]), ("a chilled drink", query_cache["a chilled drink"]),
                  ("learn to play the guitar", off_cache["learn to play the guitar"]),
                  ("the history of the Roman empire", off_cache["the history of the Roman empire"])]:
    print(f"{text:<32}", search(vec, min_score=0.3))

seafood                          [('Grilled salmon', 0.502), ('Sushi rolls', 0.489), ('Fish tacos', 0.473), ('Moules marinieres', 0.467), ('Margherita pizza', 0.426)]
a chilled drink                  [('Mango lassi', 0.469), ('Vanilla ice cream', 0.453), ('Lemon sorbet', 0.369), ('Chicken noodle soup', 0.364), ('Gazpacho', 0.363)]
learn to play the guitar         []
the history of the Roman empire  []


The off-topic queries now get an **empty list**, which the app can turn into "no recipes match".
The on-topic ones still show 5 each: here the limit decides. Note what the threshold does *not*
do: Margherita pizza (0.426) stays in the seafood list. A floor removes junk queries, not weak
hits inside a good one.

### Step 9 · The threshold and the limit together
Raise the threshold and the count changes for each query. Count results with `limit=50` so only
the threshold decides:

In [14]:
for t in [0.3, 0.4, 0.5, 0.6]:
    counts = {text: len(search(vec, k=50, min_score=t)) for text, vec in
              [("comfort food", q), ("seafood", query_cache["seafood"]), ("Italian pasta", query_cache["Italian pasta"])]}
    print(f"threshold {t}: {counts}")

threshold 0.3: {'comfort food': 35, 'seafood': 34, 'Italian pasta': 30}
threshold 0.4: {'comfort food': 11, 'seafood': 8, 'Italian pasta': 10}
threshold 0.5: {'comfort food': 1, 'seafood': 1, 'Italian pasta': 5}
threshold 0.6: {'comfort food': 0, 'seafood': 0, 'Italian pasta': 3}


At 0.3, 30 to 35 of the 50 recipes pass for each query, so 0.3 alone never makes a short list. At
0.6 "comfort food" and "seafood" return nothing, while "Italian pasta" keeps 3. A strict threshold
silently empties some good queries. So use a threshold as a floor that removes junk (0.3 here),
and let `limit` set the list length.

### Step 10 · A threshold belongs to one model
The 0.3 came from MiniLM's scores. Load the same recipes embedded with bge-small (lesson 05) into
a throwaway collection, and count hits above 0.5 for lesson 05's five queries under each model:

In [15]:
bge = np.load("../data/recipe_embeddings_bge-small.npz")
lab = QdrantClient(":memory:")
lab.create_collection("recipes_bge", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
lab.upsert("recipes_bge", points=[models.PointStruct(id=int(row.id), vector=bge["doc_emb"][i].tolist(),
           payload={"title": str(row.title)}) for i, row in enumerate(recipes.itertuples())])
for text, vec in zip(bge["queries"].tolist(), bge["query_emb"]):
    n_bge = len(lab.query_points("recipes_bge", query=vec.tolist(), limit=50, score_threshold=0.5).points)
    print(f"{text:<34} bge-small: {n_bge:>2} of 50   MiniLM: {len(search(query_cache[text], k=50, min_score=0.5)):>2} of 50")

something warm for a cold day      bge-small: 49 of 50   MiniLM:  0 of 50
a cold dessert for summer          bge-small: 50 of 50   MiniLM:  1 of 50
very spicy food                    bge-small: 50 of 50   MiniLM:  4 of 50


something to eat in the morning    bge-small: 50 of 50   MiniLM:  2 of 50
quick vegetarian dinner            bge-small: 49 of 50   MiniLM:  2 of 50


With bge-small, 0.5 lets through 49 or 50 of the 50 recipes; with MiniLM, 0 to 4. bge-small's
scores simply sit higher. Change the embedding model and you must choose the threshold again, from
data, the way Steps 6–7 did. Lesson 16 shows how to measure this properly.

### Step 11 · Thresholds on a distance metric
Lesson 08 showed that on a Euclid collection the score is a **distance**: smaller is closer. The
Qdrant docs say `score_threshold` follows the metric: "higher scores of Euclidean metric are
considered more distant and, therefore, will be excluded." So there it is a *maximum distance*.

In [16]:
lab.create_collection("recipes_euclid", vectors_config=models.VectorParams(size=384, distance=models.Distance.EUCLID))
lab.upsert("recipes_euclid", points=points)
sea = query_cache["seafood"].tolist()
for t in [0.5, 1.0, 1.1]:
    hits = lab.query_points("recipes_euclid", query=sea, limit=10, score_threshold=t).points
    print(f"max distance {t}:", [(h.payload["title"], round(h.score, 3)) for h in hits])

max distance 0.5: []
max distance 1.0: [('Grilled salmon', 0.998)]
max distance 1.1: [('Grilled salmon', 0.998), ('Sushi rolls', 1.011), ('Fish tacos', 1.027), ('Moules marinieres', 1.032), ('Margherita pizza', 1.072), ('Shrimp pad thai', 1.072), ('Guacamole', 1.088), ('Beef stew', 1.093), ('Vegetable samosas', 1.097), ('Shakshuka', 1.099)]


A larger value lets **more** results in, the opposite of Cosine. For unit vectors the two are
linked: distance = √(2 − 2·cosine) (lesson 08). So the Cosine floor 0.3 equals a Euclid ceiling
of √1.4 ≈ 1.183, and both give the same hits:

In [17]:
cos_floor = 0.3
dist_ceiling = float(np.sqrt(2 - 2 * cos_floor))
by_cos = [h.id for h in client.query_points("recipes", query=sea, limit=50, score_threshold=cos_floor).points]
by_dist = [h.id for h in lab.query_points("recipes_euclid", query=sea, limit=50, score_threshold=dist_ceiling).points]
print(f"Euclid ceiling for cosine {cos_floor}: {dist_ceiling:.3f}")
print("Cosine hits:", len(by_cos), "| Euclid hits:", len(by_dist), "| same ids, same order:", by_cos == by_dist)

Euclid ceiling for cosine 0.3: 1.183
Cosine hits: 34 | Euclid hits: 34 | same ids, same order: True


## What just happened
- Step 2: `query_points(..., limit=3)` returned a `QueryResponse` whose `.points` held `ScoredPoint`s; no `limit` meant 10.
- Step 3: Cosine scores equalled NumPy dot products; all 50 recipes got a score, down to 0.100.
- Steps 4–5: smaller k gave the start of the same list; `query=1` searched with Tomato soup's vector, leaving it out.
- Step 6: wrong answers (Beef stew 0.559) outscored right ones (Grilled salmon 0.502): scores are not comparable between queries.
- Steps 7–9: off-topic queries topped out at 0.07–0.24, so `score_threshold=0.3` emptied them and kept every on-topic query; at 0.6, "comfort food" and "seafood" were emptied too.
- Steps 10–11: 0.5 meant something else under bge-small (49–50 hits vs MiniLM's 0–4), and on Euclid the threshold is a maximum distance (√(2 − 2·0.3) ≈ 1.183 matched Cosine 0.3).

## Common mistakes

**1. Looping over the response instead of `.points`.** `QueryResponse` is a pydantic object;
looping over it yields its fields as `(name, value)` pairs, not hits:

In [18]:
try:
    response = client.query_points("recipes", query=q.tolist(), limit=3)
    for hit in response:
        print(hit.score, hit.payload["title"])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AttributeError: 'tuple' object has no attribute 'score'


It prints `AttributeError: 'tuple' object has no attribute 'score'`. That tuple is
`('points', [...])`. Fix: loop over `response.points`.

In [19]:
response = client.query_points("recipes", query=q.tolist(), limit=3)
for hit in response.points:
    print(round(hit.score, 3), hit.payload["title"])

0.537 Chicken noodle soup
0.435 Gazpacho
0.422 Ramen


**2. Reading a Euclid threshold as "similar enough".** Someone moves to the Euclid collection,
keeps "at least 0.5 means a good match", and takes the first hit:

In [20]:
try:
    hits = lab.query_points("recipes_euclid", query=sea, limit=5, score_threshold=0.5).points
    best = hits[0]
    print("best match:", best.payload["title"])
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

IndexError: list index out of range


It prints `IndexError: list index out of range`: on Euclid, 0.5 means "distance at most 0.5", and
no recipe is that close (Step 11). Fix: convert the cosine floor to a distance ceiling, and
always handle an empty result, because any threshold can return nothing.

In [21]:
hits = lab.query_points("recipes_euclid", query=sea, limit=5,
                        score_threshold=float(np.sqrt(2 - 2 * 0.3))).points
if hits:
    print("best match:", hits[0].payload["title"], round(hits[0].score, 3))
else:
    print("no recipes match")

best match: Grilled salmon 0.998


## Try it
1. **A "did you mean cooking?" message.** Using `search`, write `answer(vec)` that returns the top
   3 titles with `min_score=0.3`, or the text `"Sorry, no recipes match that."` when the list is
   empty. Try it on "a dip for chips" and on "quarterly tax return deadline".
2. **Pick a Euclid ceiling.** For "very spicy food", find the Euclid `score_threshold` that gives
   the same results as a Cosine `score_threshold=0.5` with `limit=50`, and check the ids match.

In [22]:
# your code here

Close both clients so the local folder is released.

In [23]:
lab.close()
client.close()
print("closed")

closed


## Key terms
- **query (similarity query)** — asking a collection for the stored points closest to a query vector (or to a stored point's id); in qdrant-client, `query_points`.
- **QueryResponse** — what `query_points` returns; the ranked hits are in its `.points` list.
- **ScoredPoint** — one search hit: id, payload, optional vector, `version` and the `score` against the query.
- **score** — the number the collection's metric gives a query and a stored vector: a similarity for Cosine/Dot (higher is closer), a distance for Euclid/Manhattan (lower is closer). It describes two vectors, not relevance, and its scale depends on the model and the query.
- **limit** — the k in top-k: the most results a query returns (default 10); never more than the collection holds.
- **score threshold** — `score_threshold`: drop hits whose score is worse than this value (below it for Cosine/Dot, above it for Euclid/Manhattan); chosen from data, per model.